# Embeddings Analysis
This notebook will help visualization of the embeddings using PCA.
This notebook is generated with Claude Haiku 4.5

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image
import random
from matplotlib import patches
from pprint import pprint
import warnings
warnings.filterwarnings('ignore')
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from dotenv import load_dotenv
# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("✓ All libraries imported successfully")

In [ ]:
from utils.dataloader import *
from model.models import SimCLRModel

# Parameters

In [ ]:
load_dotenv()  
DATASET_PATH = os.getenv('DATASET_PATH')
manga_name_list = get_book_list(DATASET_PATH)

TARGET_SIZE = (112,112)

trial_no = 5

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device", device)

## Load Model

In [ ]:
model = SimCLRModel()
model.load_state_dict(torch.load(f'simclr-{trial_no}.pt', weights_only=True, map_location=torch.device('cpu')))
model.eval()


## Load Dataset

In [ ]:
################################
#   Edit manga name here.
################################
manga_name = 'KoukouNoHitotachi'

annotations = annotation_loader(DATASET_PATH, manga_name)
pages = annotations['book']['pages']['page']
char_id, _ = get_character_list(annotations)
temp_char_dict = create_data(DATASET_PATH, annotations, TARGET_SIZE)
character_dict = {d['id']:d['face_imgs'] for d in temp_char_dict.values()}

In [ ]:
# Create dictionary with character id -> num images
char_counts = {char_id: len(images) for char_id, images in character_dict.items()}

# Pretty print the dictionary
pprint(list(char_counts.items()), width=100, compact=True)

print(f"\nTotal characters: {len(char_counts)}")
print(f"Total images: {sum(char_counts.values())}")

In [ ]:
# ========== USER SELECTION ==========
# Modify this list to select your desired character IDs
selected_chars = list(key for key, value in char_counts.items() if value > 30)
# ====================================

# Verify selected characters exist
valid_chars = [char_id for char_id in selected_chars if char_id in character_dict]
invalid_chars = [char_id for char_id in selected_chars if char_id not in character_dict]

if invalid_chars:
    print(f"\n⚠ Warning: The following characters do not exist: {invalid_chars}")
    print(f"  These will be skipped.")

selected_chars = valid_chars

print(f"\n✓ Selected characters:")
for char_id in selected_chars:
    count = char_counts[char_id]
    print(f"  - {char_id}: {count} images")

In [ ]:
# ========== USER SELECTION ==========
images_per_character = 40  # Number of images per character (k)
# ====================================

print(f"\nNumber of images per character: {images_per_character}")

# Select k images for each character
selected_data = {}
char_to_images = {}  # Store for later visualization

all_images = []
all_labels = []

for char_id in selected_chars:
    available_images = character_dict[char_id]
    num_to_select = min(images_per_character, len(available_images))
    
    # Randomly select images
    selected_images = random.sample(available_images, num_to_select)
    selected_data[char_id] = selected_images
    char_to_images[char_id] = selected_images
    
    # Add to flat lists
    all_images.extend(selected_images)
    all_labels.extend([char_id] * num_to_select)
    
    print(f"  {char_id}: {num_to_select}/{len(available_images)} images selected")

print(f"\n✓ Total images to process: {len(all_images)}")

# Extract Embeddings

In [ ]:

def extract_embeddings(images, model, device, batch_size=32, image_size=(224, 224)):
    '''
    Extract embeddings for a list of images
    '''
    embeddings = []
    
    model = model.to(device)
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch_images = images[i:i+batch_size]
            
            # Preprocess batch
            batch_tensors = []
            for img in batch_images:
                batch_tensors.append(simple_transform(img))
            
            batch_tensor = torch.stack(batch_tensors).to(device)
            batch_embeddings = model(batch_tensor)
            embeddings.append(batch_embeddings.cpu().detach().numpy())
    
    embeddings = np.vstack(embeddings)
    return embeddings

In [ ]:
embeddings = extract_embeddings(
    all_images, 
    model, 
    device, 
    batch_size=32, 
    image_size=TARGET_SIZE
)

## PCA

In [ ]:
# Standardize embeddings
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

# Perform PCA - keep most components
n_components = min(len(all_images) - 1, embeddings.shape[1])
pca = PCA(n_components=n_components)
embeddings_pca = pca.fit_transform(embeddings_scaled)

# Calculate explained variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print(f"\n✓ PCA completed")
print(f"  Total components: {len(explained_variance)}")
print(f"  Top 5 components explain: {cumulative_variance[min(4, len(explained_variance)-1)]:.2%} of variance")

print(f"\nExplained variance for top 10 components:")
for i in range(min(10, len(explained_variance))):
    print(f"  PC{i+1}: {explained_variance[i]:.4f} ({cumulative_variance[i]:.2%})")


In [ ]:
# ========== USER SELECTION ==========
dim1 = 0  # First dimension (0-indexed, so 0 = PC1)
dim2 = 2  # Second dimension (0-indexed, so 1 = PC2)
# ====================================

print(f"\nSelected dimensions: PC{dim1+1} and PC{dim2+1}")

# Verify dimensions are valid
if dim1 >= embeddings_pca.shape[1] or dim2 >= embeddings_pca.shape[1]:
    print("⚠ Invalid dimension selection. Using PC1 and PC2.")
    dim1, dim2 = 0, 1

# Extract selected dimensions
data_2d = embeddings_pca[:, [dim1, dim2]]

# Get variance for selected dimensions
var_dim1 = explained_variance[dim1]
var_dim2 = explained_variance[dim2]

print(f"  PC{dim1+1} explains: {var_dim1:.2%} of variance")
print(f"  PC{dim2+1} explains: {var_dim2:.2%} of variance")
print(f"  Combined: {var_dim1 + var_dim2:.2%} of variance")

In [ ]:
# Define colors and markers for each character
colors = plt.cm.tab10(np.linspace(0, 1, len(selected_chars)))
markers = ['o', 's', '^', 'D', 'v', 'p', '*', 'h', 'X', '+']

fig, ax = plt.subplots(figsize=(12, 12))

# Plot each character with different marker and color
for idx, char_id in enumerate(selected_chars):
    # Get indices for this character
    char_indices = [i for i, label in enumerate(all_labels) if label == char_id]
    
    x = data_2d[char_indices, 0]
    y = data_2d[char_indices, 1]
    
    ax.scatter(x, y, 
              c=[colors[idx]], 
              marker=markers[idx % len(markers)],
              s=150, 
              alpha=0.7, 
              label=f'{char_id} (n={len(char_indices)})',
              edgecolors='black',
              linewidth=1.5)

# Labels and title
ax.set_xlabel(f'PC{dim1+1} ({var_dim1:.2%})', fontsize=12, fontweight='bold')
ax.set_ylabel(f'PC{dim2+1} ({var_dim2:.2%})', fontsize=12, fontweight='bold')
ax.set_title(f'Character Embeddings Visualization (PCA)', fontsize=14, fontweight='bold')

# Add grid
ax.grid(True, alpha=0.3, linestyle='--')

# Legend
ax.legend(loc='best', fontsize=10, framealpha=0.9)

plt.tight_layout()
plt.show()

## Sample image

In [ ]:
# Number of samples to display per character
num_samples = min(5, images_per_character)

fig = plt.figure(figsize=(16, len(selected_chars) * 3))

for row_idx, char_id in enumerate(selected_chars):
    images_to_show = char_to_images[char_id][:num_samples]
    
    for col_idx, img in enumerate(images_to_show):
        ax = plt.subplot(len(selected_chars), num_samples, 
                        row_idx * num_samples + col_idx + 1)
        
        # Convert image to proper format for display
        if isinstance(img, np.ndarray):
            if img.max() <= 1.0:
                img_display = img
            else:
                img_display = img.astype(np.float32) / 255.0
        else:
            img_display = img
        
        ax.imshow(img_display)
        
        # Title for first column
        if col_idx == 0:
            ax.set_ylabel(f'{char_id}', fontsize=12, fontweight='bold')
        
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect('auto')

plt.suptitle(f'Sample Images for Each Character (showing {num_samples} per character)', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print(f"\n✓ Sample images displayed")

# Show YOLO-retrieved images

In [ ]:
from yolo_8 import *
## This may take a few minutes
pages = annotations['book']['pages']['page']
print(len(pages))

annotation_boxes_count = np.sum([
    len(page.get('face')) for page in pages if 'face' in page
])

char_id, _ = get_character_list(annotations)

all_cropped_chars = {}

for page_num in range(len(pages)):
    page_img = retrieve_page(DATASET_PATH, manga_name, page_num)
    page_boxes = get_crop_boxes_from_yolo(page_img)[0]
    all_positions = [crop_info['position'] for crop_info in page_boxes]

    results = get_paired_pred_boxes_labels(all_positions, annotations, page_num)

    for result in results:
        char_list = all_cropped_chars.setdefault(result[1], [])
        char_list.append(
        retrieve_page(DATASET_PATH, manga_name, page_num, result[0], target_size=TARGET_SIZE))

In [ ]:
# ========== USER SELECTION ==========
# Choose which of the previously selected characters to show in new data
selected_chars_new = ['00000473', '000004c0', '00000657']
new_images_per_character = 10

# Verify selected characters exist in new data
valid_chars_new = [char_id for char_id in selected_chars_new 
                   if char_id in all_cropped_chars]
invalid_chars_new = [char_id for char_id in selected_chars_new 
                     if char_id not in all_cropped_chars]

if invalid_chars_new:
    print(f"⚠ Warning: The following characters not in new data: {invalid_chars_new}")
    print(f"  These will be skipped.")

selected_chars_new = valid_chars_new

print(f"\nNumber of images per character from new data: {new_images_per_character}")

# Select images for new data
new_selected_data = {}
new_char_to_images = {}

new_all_images = []
new_all_labels = []

for char_id in selected_chars_new:
    available_images = all_cropped_chars[char_id]
    num_to_select = min(new_images_per_character, len(available_images))
    
    # Randomly select images
    selected_images = random.sample(available_images, num_to_select)
    new_selected_data[char_id] = selected_images
    new_char_to_images[char_id] = selected_images
    
    # Add to flat lists
    new_all_images.extend(selected_images)
    new_all_labels.extend([char_id] * num_to_select)
    
    print(f"  {char_id}: {num_to_select}/{len(available_images)} images selected")

print(f"\n✓ Total new images to process: {len(new_all_images)}")


In [ ]:
new_embeddings = extract_embeddings(new_all_images, model, device, 
                                     batch_size=32, image_size=TARGET_SIZE)
# Standardize new embeddings using the same scaler fitted on original data
new_embeddings_scaled = scaler.transform(new_embeddings)

# Transform using the fitted PCA model
new_embeddings_pca = pca.transform(new_embeddings_scaled)

print(f"\n✓ New embeddings projected onto PCA space")
print(f"  Shape after PCA: {new_embeddings_pca.shape}")

# Extract selected dimensions for new data
new_data_2d = new_embeddings_pca[:, [dim1, dim2]]

In [ ]:
# Define colors and markers
# For characters that appear in both old and new data
colors_old = plt.cm.Blues(np.linspace(0.4, 0.8, len(selected_chars_new)))
colors_new = plt.cm.Reds(np.linspace(0.4, 0.8, len(selected_chars_new)))
markers = ['o', 's', '^', 'D', 'v', 'p', '*', 'h', 'X', '+']

fig, ax = plt.subplots(figsize=(14, 10))

# Create a mapping of character IDs to their index for consistent coloring
char_to_idx = {char_id: idx for idx, char_id in enumerate(selected_chars_new)}

# Plot OLD data
print("\nPlotting OLD data:")
for idx, char_id in enumerate(selected_chars_new):
    # Get indices for this character in old data
    char_indices = [i for i, label in enumerate(all_labels) if label == char_id]
    
    if char_indices:  # Only plot if character exists in old data
        x = data_2d[char_indices, 0]
        y = data_2d[char_indices, 1]
        
        ax.scatter(x, y, 
                  c=[colors_old[idx]], 
                  marker=markers[idx % len(markers)],
                  s=150, 
                  alpha=0.7, 
                  label=f'{char_id} (OLD, n={len(char_indices)})',
                  edgecolors='black',
                  linewidth=1.5)
        
        print(f"  ✓ {char_id}: {len(char_indices)} samples")

# Plot NEW data
print("\nPlotting NEW data:")
for idx, char_id in enumerate(selected_chars_new):
    # Get indices for this character in new data
    char_indices = [i for i, label in enumerate(new_all_labels) if label == char_id]
    
    if char_indices:  # Only plot if character exists in new data
        x = new_data_2d[char_indices, 0]
        y = new_data_2d[char_indices, 1]
        
        ax.scatter(x, y, 
                  c=[colors_new[idx]], 
                  marker=markers[idx % len(markers)],
                  s=150, 
                  alpha=0.7, 
                  label=f'{char_id} (NEW, n={len(char_indices)})',
                  edgecolors='darkred',
                  linewidth=1.5,
                  linestyle=':')
        
        print(f"  ✓ {char_id}: {len(char_indices)} samples")

# Labels and title
ax.set_xlabel(f'PC{dim1+1} ({var_dim1:.2%})', fontsize=12, fontweight='bold')
ax.set_ylabel(f'PC{dim2+1} ({var_dim2:.2%})', fontsize=12, fontweight='bold')
ax.set_title(f'Character Embeddings: Old vs New Data', fontsize=14, fontweight='bold')

# Add grid
ax.grid(True, alpha=0.3, linestyle='--')

# Legend with two columns
ax.legend(loc='best', fontsize=9, framealpha=0.95, ncol=2)

# Add text box explaining the visualization
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)

plt.tight_layout()
plt.show()

print(f"\n✓ Combined scatter plot created successfully")

In [ ]:
num_samples_display = min(3, images_per_character, new_images_per_character)

fig = plt.figure(figsize=(18, len(selected_chars_new) * 3))

for row_idx, char_id in enumerate(selected_chars_new):
    # OLD DATA
    old_images = char_to_images[char_id][:num_samples_display]
    for col_idx, img in enumerate(old_images):
        ax = plt.subplot(len(selected_chars_new), 
                        2 * num_samples_display, 
                        row_idx * (2 * num_samples_display) + col_idx + 1)
        
        if isinstance(img, np.ndarray):
            if img.max() <= 1.0:
                img_display = img
            else:
                img_display = img.astype(np.float32) / 255.0
        else:
            img_display = img
        
        ax.imshow(img_display)
        
        if col_idx == 0:
            ax.set_ylabel(f'{char_id}\n(OLD)', fontsize=10, fontweight='bold')
        
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect('auto')
        
        # Add border to indicate old data
        for spine in ax.spines.values():
            spine.set_edgecolor('blue')
            spine.set_linewidth(2)
    
    # NEW DATA
    new_images = new_char_to_images[char_id][:num_samples_display]
    for col_idx, img in enumerate(new_images):
        ax = plt.subplot(len(selected_chars_new), 
                        2 * num_samples_display, 
                        row_idx * (2 * num_samples_display) + num_samples_display + col_idx + 1)
        
        if isinstance(img, np.ndarray):
            if img.max() <= 1.0:
                img_display = img
            else:
                img_display = img.astype(np.float32) / 255.0
        else:
            img_display = img
        
        ax.imshow(img_display)
        
        if col_idx == 0:
            ax.set_ylabel(f'{char_id}\n(NEW)', fontsize=10, fontweight='bold', color='red')
        
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect('auto')
        
        # Add border to indicate new data
        for spine in ax.spines.values():
            spine.set_edgecolor('red')
            spine.set_linewidth(2)

plt.suptitle(f'Sample Images - Old vs New Data (showing {num_samples_display} per character)', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print(f"\n✓ Sample images comparison displayed")